In [1]:
import pandas as pd
import json
import pickle
import os
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print("Libraries imported!")

Libraries imported!


In [2]:
# Load chunks
chunks_df = pd.read_csv('data/chunks_512.csv')
corpus = chunks_df['text'].tolist()

# Load test questions
with open('data/test_questions.json', 'r') as f:
    test_questions = json.load(f)

# Load BM25
with open('src/retrievers/bm25_index.pkl', 'rb') as f:
    bm25 = pickle.load(f)

# Load FAISS
index = faiss.read_index('src/retrievers/faiss_index.bin')

# Load embeddings model
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Chunks: {len(corpus)}")
print(f"Questions: {len(test_questions)}")
print(f"FAISS vectors: {index.ntotal}")
print("All loaded!")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks: 3725
Questions: 20
FAISS vectors: 3725
All loaded!


In [3]:
def bm25_retrieve(query, top_k=20):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = scores.argsort()[-top_k:][::-1]
    return [(idx, scores[idx]) for idx in top_indices]

def dense_retrieve(query, top_k=20):
    query_vector = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(
        query_vector.astype('float32'), top_k
    )
    return [(indices[0][i], distances[0][i]) for i in range(top_k)]

print("Retriever functions ready!")

Retriever functions ready!


In [4]:
def hybrid_retrieve(query, top_k=5, bm25_weight=0.4, dense_weight=0.6):
    # Get results from both retrievers
    bm25_results = bm25_retrieve(query, top_k=20)
    dense_results = dense_retrieve(query, top_k=20)
    
    # Normalize BM25 scores (higher is better)
    bm25_scores = [s for _, s in bm25_results]
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1
    
    # Normalize Dense distances (lower is better → invert)
    dense_distances = [d for _, d in dense_results]
    max_dense = max(dense_distances) if max(dense_distances) > 0 else 1
    
    # Combine scores
    combined = {}
    
    for idx, score in bm25_results:
        normalized = score / max_bm25
        combined[idx] = combined.get(idx, 0) + bm25_weight * normalized
    
    for idx, dist in dense_results:
        normalized = 1 - (dist / max_dense)  # invert distance
        combined[idx] = combined.get(idx, 0) + dense_weight * normalized
    
    # Sort by combined score
    sorted_results = sorted(
        combined.items(), 
        key=lambda x: x[1], 
        reverse=True
    )[:top_k]
    
    results = []
    for idx, score in sorted_results:
        results.append({
            'chunk_id': chunks_df.iloc[idx]['chunk_id'],
            'text': corpus[idx],
            'combined_score': score
        })
    
    return results

print("Hybrid retriever ready!")

Hybrid retriever ready!


In [5]:
print("=== TESTING HYBRID RETRIEVER ===\n")

for i, item in enumerate(test_questions[:5]):
    query = item['question']
    results = hybrid_retrieve(query, top_k=3)
    
    print(f"Question {i+1}: {query}")
    print(f"Top combined score: {results[0]['combined_score']:.4f}")
    print(f"Top result preview: {results[0]['text'][:150]}...")
    print("-" * 50)

=== TESTING HYBRID RETRIEVER ===

Question 1: What type of system is being analyzed in the paper for the mean resolvent using a polymer expansion?
Top combined score: 0.5861
Top result preview: in this paper we develop a polymer expansion with large / small field conditions for the mean resolvent of a weakly disordered system . then we show t...
--------------------------------------------------
Question 2: What is the significance of the asymptotic expansion for the density of states in the context of the research paper?
Top combined score: 0.5275
Top result preview: in this paper we develop a polymer expansion with large / small field conditions for the mean resolvent of a weakly disordered system . then we show t...
--------------------------------------------------
Question 3: What is the asymptotic long-time equivalence being referred to in the context of the paper?
Top combined score: 0.4574
Top result preview: we show the asymptotic long - time equivalence of a generic power law

In [6]:
print("Evaluating Hybrid Retriever...\n")

hybrid_results = []

for item in tqdm(test_questions):
    query = item['question']
    results = hybrid_retrieve(query, top_k=5)
    
    source = item['source_abstract']
    retrieved_texts = [r['text'] for r in results]
    
    hit = any(
        len(set(source.lower().split()) & 
            set(r.lower().split())) > 20 
        for r in retrieved_texts
    )
    
    hybrid_results.append({
        'question': query,
        'hit': bool(hit),
        'top_score': float(results[0]['combined_score']),
        'top_result': results[0]['text']
    })

hits = sum(1 for r in hybrid_results if r['hit'])
accuracy = hits / len(hybrid_results) * 100

print(f"Hybrid Retriever Results:")
print(f"Total questions: {len(hybrid_results)}")
print(f"Correct retrievals: {hits}")
print(f"Accuracy: {accuracy:.1f}%")

Evaluating Hybrid Retriever...



100%|██████████| 20/20 [00:01<00:00, 12.06it/s]

Hybrid Retriever Results:
Total questions: 20
Correct retrievals: 20
Accuracy: 100.0%


In [7]:
with open('results/hybrid_results.json', 'w') as f:
    json.dump(hybrid_results, f, indent=2)

summary = {
    'retriever': 'Hybrid (BM25 + FAISS)',
    'total_questions': len(hybrid_results),
    'hits': hits,
    'accuracy': float(accuracy),
    'bm25_weight': 0.4,
    'dense_weight': 0.6
}

with open('results/hybrid_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Results saved!")

Results saved!


In [8]:
with open('results/bm25_summary.json', 'r') as f:
    bm25_sum = json.load(f)

with open('results/dense_summary.json', 'r') as f:
    dense_sum = json.load(f)

print("=" * 50)
print("      FINAL RETRIEVER COMPARISON TABLE")
print("=" * 50)
print(f"{'Retriever':<20} {'Accuracy':>10} {'Hits':>8}")
print("-" * 50)
print(f"{'BM25 (Sparse)':<20} {bm25_sum['accuracy']:>9.1f}% {bm25_sum['hits']:>8}")
print(f"{'Dense (FAISS)':<20} {dense_sum['accuracy']:>9.1f}% {dense_sum['hits']:>8}")
print(f"{'Hybrid':<20} {accuracy:>9.1f}% {hits:>8}")
print("=" * 50)

      FINAL RETRIEVER COMPARISON TABLE
Retriever              Accuracy     Hits
--------------------------------------------------
BM25 (Sparse)            100.0%       20
Dense (FAISS)            100.0%       20
Hybrid                   100.0%       20
